**OSTIn project: Develop software to automatically launch infrasound and satellite imagery analyses using the time and location provided by a VONA (Volcano Observatory Notice for Aviation).**




**Parse VONA messages to extract time and location data from both MAGMA (MAGMA VONA) and TVAAC (TVAAC VONA) in real time** 

Sources:
MAGMA VONA: https://magma.esdm.go.id/v1/vona 

TVAAC VONA: https://www.data.jma.go.jp/svd/vaac/data/vaac_list.html 

**Forecast Eruption Source Paramerters such as Mass Eruption Rate, Total Erupted Mass, Eruption Duration and percentage of fine ash particles based on VONA messages and Historical Volcanic Eruption Data using Bayesian statistics**


**VONA Monitoring Script**

This script continuously monitors the latest Volcanic Observatory Notice for Aviation (VONA) reports 
and detects changes in key volcanic activity parameters such as volcano name, ash cloud height above ground level (AGL),
and volcanic explosivity index (VEI) range. When changes are detected, the screen is cleared, and updated information is displayed.

Key Features:
- Fetches VONA reports from a specified data source (default: "TOKYO").
- Detects changes in volcano name, ash cloud height, and VEI range.
- Updates the display when changes occur.
- Uses the `ESP_INIT` function for eruption source parameter analysis when new data is available.
- Implements a progress bar to visualize the monitoring interval.

Dependencies:
- `vona_trigger`: Retrieves VONA reports.
- `re`: For text normalization.
- `esp_init.ESP_INIT`: Processes eruption source parameters.
- `time`: Introduces delays between monitoring cycles.
- `os`: Clears the terminal screen.
- `tqdm`: Provides a visual progress bar for monitoring intervals.

Contact: 
Dr. Mahmud Muhammad
Email: mahmud.muhammad@hotmail.com



In [ ]:
"""
VONA Monitoring Script

This script continuously monitors the latest Volcanic Observatory Notice for Aviation (VONA) reports 
and detects changes in key volcanic activity parameters such as volcano name, ash cloud height above ground level (AGL),
and volcanic explosivity index (VEI) range. When changes are detected, the screen is cleared, and updated information is displayed.

Key Features:
- Fetches VONA reports from a specified data source (default: "MAGMA").
- Detects changes in volcano name, ash cloud height, and VEI range.
- Updates the display when changes occur.
- Uses the `ESP_INIT` function for eruption source parameter analysis when new data is available.
- Implements a progress bar to visualize the monitoring interval.

Dependencies:
- `vona_trigger`: Retrieves VONA reports.
- `re`: For text normalization.
- `esp_init.ESP_INIT`: Processes eruption source parameters.
- `time`: Introduces delays between monitoring cycles.
- `os`: Clears the terminal screen.
- `tqdm`: Provides a visual progress bar for monitoring intervals.

Contact: 
Dr. Mahmud Muhammad
Email: mahmud.muhammad@hotmail.com

"""


import vona_trigger
import re
from esp_init import ESP_INIT
import time
import os
from tqdm import tqdm  # For progress bar
from vona_trigger import PlumeHeightAnalyzer_FromInfrasound

### Plume Height from Infrasound
analyzer = PlumeHeightAnalyzer_FromInfrasound("./allPlumeHeights/")
dataset_name = "sparks"  # sparks or mastin

mono_mean = analyzer.get_mono_mean(dataset_name) # This represent min Plume Height
di_mean = analyzer.get_di_mean(dataset_name) # This represents max plume Height 

print(f"Mono Mean for {dataset_name}: {mono_mean}")
print(f"DI Mean for {dataset_name}: {di_mean}")

combined= None

Use_Infrasound=False # Change this to True if you want to use observed Plume Height from Infrasound Team
if Use_Infrasound:
    combined= mono_mean + di_mean
    combined = list(set(int(round(num)) for num in combined))
    # Sort the result (optional)
    combined.sort()

Use_Infrasound_Height = combined if combined is not None else None
print(f"Combined Mono and di mean Plume Heights: {combined}")

########################################

# Initialize storage for previous variable values
previous_values = {
    "volcano_name": None,
    "AshCloudHeight_AGL": None,
    "vei_range": None
}

def clear_screen():
    """Clears the terminal screen for a fresh output."""
    os.system('cls' if os.name == 'nt' else 'clear')

def normalize_text(text):
    """Removes extra spaces and normalizes text for meaningful comparison."""
    if isinstance(text, str):
        return re.sub(r'\s+', ' ', text).strip()  # Replace multiple spaces with one and trim
    return text  # Return non-string values as-is


while True:
    # Fetch the latest VONA data
    VONA_reports, VONA_latest_data, volcano_name, latitude, longitude, AshCloudHeight_AGL, vei_range = vona_trigger.VONA_Message(VONA_Source="MAGMA")
   
        
    
    # Store current values in a dictionary
    current_values = {
       
        "volcano_name": volcano_name,
        "AshCloudHeight_AGL": AshCloudHeight_AGL,
        "vei_range": vei_range
    }

   # Check for changes in each variable
    changes_detected = False
    for key in current_values:
        prev_value = normalize_text(previous_values[key]) if previous_values[key] is not None else None
        curr_value = normalize_text(current_values[key])
        
        if prev_value != curr_value:
            changes_detected = True
    
    # If changes are detected, clear screen and print new data
    
    if changes_detected:
        clear_screen()  # Clears previous prints and outputs
        print("🔔 **New VONA Data Detected:**\n")
        for key in current_values:
            prev_value = normalize_text(previous_values[key]) if previous_values[key] is not None else None
            curr_value = normalize_text(current_values[key])
            
            if prev_value != curr_value:
                print(f"🔹 **Change detected in {key}:**")
                print(f"   - **Previous:** {prev_value}")
                print(f"   - **New:** {curr_value}\n")
                previous_values[key] = current_values[key]  # Update stored value

        # Convert AshCloudHeight_AGL to km if it's not None
        if AshCloudHeight_AGL is not None or vei_range is  None:
            
            heights=ESP_INIT(vei_levels=None, PlumeHeight=AshCloudHeight_AGL, show_plots=True)

        if vei_range is not None or AshCloudHeight_AGL is  None: 
            heights=ESP_INIT(vei_levels=vei_range, PlumeHeight=None, show_plots=True)
        
        # if AshCloudHeight_AGL is not None and vei_range is not None:
        
        #     ESP_INIT(vei_levels=None, PlumeHeight=AshCloudHeight_AGL, show_plots=True)
        #     ESP_INIT(vei_levels=vei_range, PlumeHeight=None, show_plots=True)
        
        if AshCloudHeight_AGL is  None and vei_range is  None:
            heights=ESP_INIT(vei_levels=None, PlumeHeight=None, show_plots=True)
   
        if Use_Infrasound_Height is not None:
            heights=ESP_INIT(vei_levels=None, PlumeHeight=Use_Infrasound_Height, show_plots=True)
            analyzer.visualize_datasets()
    
    if heights is not None:
    
        from nameModel_file_generator import SimulationParameterGenerator

        generator = SimulationParameterGenerator(
                vona_input=VONA_latest_data,
                plume_input=heights ,
                output_dir="./simulation_outputs"
            )

        created_files = generator.run()

        print("Simulation parameter files created:")
        for path in created_files:
            print(path)
    
    
    # Progress bar with dynamic status
    status_message = "🟢 Waiting for next VONA report..." if changes_detected else "🟡 No new data - Monitoring..."
    
    #############
    
    VONA_latest_data.head()
    
    VONA_latest_data.to_csv('vona_latest_data.csv')
    
    #############

    with tqdm(total=1800, desc=status_message, unit="s", bar_format="{l_bar}{bar} {remaining}") as pbar:
        for _ in range(1800):
            time.sleep(1)  # Sleep in 1-second intervals
            pbar.update(1)  # Update progress bar



Mono Mean for sparks: [8.29346551961875, 15.056916546925, 15.43322716493125, 15.784978183993749, 16.115633107675, 16.4279436496625, 16.724135575693747, 9.924366913831252, 11.023270081037499, 11.875983376899999, 12.58256766244375, 13.190984723050002, 13.72828971745, 14.2113831939875, 14.651593243575]
DI Mean for sparks: [10.6103734279625, 19.26329914673125, 19.74473796040625, 20.194756073687497, 20.61778455806875, 21.017343888875004, 21.396281605318748, 12.69689236679375, 14.102791137118752, 15.1937230842875, 16.09770261904375, 16.87609039804375, 17.563499853037502, 18.18155296958125, 18.744742507774998]
Combined Mono and di mean Plume Heights: None
CSV file created: ./TVAAC_test\vaa_texts.csv
Scraping up to 1 pages for All Volcanoes...


🔄 Scraping Pages: 100%|██████████| 1/1 [00:07<00:00,  7.49s/page]


Scraping completed. Data saved at Magma_Test\vona_reports_sorted.csv
Extracting details from 15 reports...


📝 Extracting Reports:   0%|          | 0/15 [00:00<?, ?report/s]2025-05-14 11:11:09,583 - WARNING - Connection pool is full, discarding connection: magma.esdm.go.id. Connection pool size: 10
2025-05-14 11:11:09,583 - WARNING - Connection pool is full, discarding connection: magma.esdm.go.id. Connection pool size: 10
📝 Extracting Reports: 100%|██████████| 15/15 [00:01<00:00,  8.36report/s]



💾 Saving extracted reports...


...

                                    VEI and Related Data                                    
┏━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━┓
┃ VEI ┃ Height_km.a.v.l_Percentile_90 ┃ Height_km.a.v.l_Percentile_95 ┃   Duration_hr_P5   ┃
┡━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━┩
│ 2.0 │       7.187988430054962       │       7.40846877292765        │ 3.6401170830821266 │
└─────┴───────────────────────────────┴───────────────────────────────┴────────────────────┘

                        VEI and Related Data                         
┏━━━━━┳━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ VEI ┃  Duration_hr_P25   ┃  Duration_hr_P50   ┃  Duration_hr_P75  ┃
┡━━━━━╇━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ 2.0 │ 2.9926755175033515 │ 2.6000613226002938 │ 2.264813658593156 │
└─────┴────────────────────┴────────────────────┴───────────────────┘

    VEI and Related Data    
┏━━━━━┳━━━━━━━━━━━━━━━━━━━━┓
┃ VEI ┃  Duration_hr_P95   ┃
┡━━━━━╇━━━━━━━━━━━━━━━━━━━━┩
│ 2.0 │ 1.8552899511851653 │
└─────┴────────────────────┘

Model updated with xvar='MER_kg/s'
Model updated with yvar='MER_kg/s'.
Model updated with yvar='MASS_FRACTION_ASH_BELOW_63_micron'.
Saved plot to ESP_init_forVEI_[2]/Forecast_MASS_FRACTION_ASH_BELOW_63_micron\posterior_predictive_plot.png
Saved plot to ESP_init_forVEI_[2]/Forecast_MASS_FRACTION_ASH_BELOW_63_micron\percentiles_plot.png
Saved plot to ESP_init_forVEI_[2]/Forecast_MASS_FRACTION_ASH_BELOW_63_micron\contour_percentiles_plot.png
Saved DataFrame to ESP_init_forVEI_[2]/Forecast_MASS_FRACTION_ASH_BELOW_63_micron\forecast_MASS_FRACTION_ASH_BELOW_63_micron.csv
        MER_kg/s  MASS_FRACTION_ASH_BELOW_63_micron_Percentile_5  \
0  531351.109117                                        0.135613   

   MASS_FRACTION_ASH_BELOW_63_micron_Percentile_10  \
0                                         0.135782   

   MASS_FRACTION_ASH_BELOW_63_micron_Percentile_15  \
0                                         0.135904   

   MASS_FRACTION_ASH_BELOW_63_micron_Percentile_20  \
0                  

🟢 Waiting for next VONA report...:  93%|█████████▎ 5:40:45  

In [ ]:
from nameModel_file_generator import SimulationParameterGenerator

generator = SimulationParameterGenerator(
        vona_input="./vona_latest_data.csv",
        plume_input="./ESP_init_forVEI_[2, 3, 4, 5, 6]/Plume_Height_results.csv",
        output_dir="./simulation_outputs_test"
    )

created_files = generator.run()

print("Simulation parameter files created:")
for path in created_files:
    print(path)


In [ ]:
from vona_trigger import PlumeHeightAnalyzer_FromInfrasound

### Plume Height from Infrasound
analyzer = PlumeHeightAnalyzer_FromInfrasound("./allPlumeHeights/")
dataset_name = "sparks"  # sparks or mastin

mono_mean = analyzer.get_mono_mean(dataset_name)
di_mean = analyzer.get_di_mean(dataset_name)

print(f"Mono Mean for {dataset_name}: {mono_mean}")
print(f"DI Mean for {dataset_name}: {di_mean}")

combined=mono_mean + di_mean

# Round numbers to two decimal places and remove duplicates
combined = list(set(int(round(num)) for num in combined))

# Sort the result (optional)
combined.sort()

print(f"Combined Mono and di mean Plume Heights: {combined}")

analyzer.visualize_datasets()